# 12. Preparación anual de base para determinantes del ingreso

Este notebook prepara una base analítica **interpretable y reproducible** para estudiar asociaciones entre características personales, laborales, del hogar y territoriales e ingreso laboral/de negocio en ENIGH.

La ejecución predeterminada es 2024. Para preparar otro año válido, cambiar únicamente `ANIO_ANALISIS` en la siguiente celda y volver a ejecutar todo el notebook.

La etapa no entrena modelos, no crea particiones, no reactiva deflactores y no usa diseño muestral formal JKn.

In [1]:
ANIO_ANALISIS = 2024
ANIOS_VALIDOS = (2018, 2020, 2022, 2024)
EDAD_MINIMA = 18

## Configuración efectiva

El año se valida explícitamente contra `ANIOS_VALIDOS`. Las rutas de salida se derivan del año elegido para evitar que una ejecución de otro año sobrescriba resultados de 2024.

In [2]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.features.preparacion_determinantes import PreparationConfig, build_outputs, validate_analysis_year

ANIO_ANALISIS = validate_analysis_year(ANIO_ANALISIS, ANIOS_VALIDOS)
CONFIG = PreparationConfig.from_root(
    PROJECT_ROOT,
    year=ANIO_ANALISIS,
    min_age=EDAD_MINIMA,
    valid_years=ANIOS_VALIDOS,
)

TABLE_DIR = CONFIG.table_dir
TABLE_ROOT = TABLE_DIR.parent
FIG_DIR = CONFIG.figure_dir
PROCESSED_DIR = CONFIG.processed_dir

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.4f}".format)

try:
    from IPython.display import Image, display
except Exception:
    Image = None
    display = None


def mostrar(df, n=20):
    vista = df.head(n)
    if display is not None:
        display(vista)
    else:
        print(vista.to_string(index=False))


configuracion = {
    "ANIO_ANALISIS": ANIO_ANALISIS,
    "ANIOS_VALIDOS": ANIOS_VALIDOS,
    "EDAD_MINIMA": EDAD_MINIMA,
    "entrada": str(CONFIG.input_path),
    "data_processed": str(PROCESSED_DIR),
    "tablas": str(TABLE_DIR),
    "figuras": str(FIG_DIR),
}
print(json.dumps(configuracion, ensure_ascii=False, indent=2))

{
  "ANIO_ANALISIS": 2024,
  "ANIOS_VALIDOS": [
    2018,
    2020,
    2022,
    2024
  ],
  "EDAD_MINIMA": 18,
  "entrada": "C:\\Users\\lucia\\OneDrive\\Escritorio\\Fer\\inegi-income-modeling\\data\\interim\\revision_4\\mart_persona_2018_2024.csv.gz",
  "data_processed": "C:\\Users\\lucia\\OneDrive\\Escritorio\\Fer\\inegi-income-modeling\\data\\processed\\determinantes_2024",
  "tablas": "C:\\Users\\lucia\\OneDrive\\Escritorio\\Fer\\inegi-income-modeling\\reports\\tables\\preparacion_determinantes\\2024",
  "figuras": "C:\\Users\\lucia\\OneDrive\\Escritorio\\Fer\\inegi-income-modeling\\reports\\figures\\preparacion_determinantes\\2024"
}


## Ejecución reproducible del año seleccionado

La celda siguiente reconstruye los artefactos para `ANIO_ANALISIS` desde el mart nominal aprobado de `revision_4`.

También inspecciona compatibilidad de esquema para los años válidos, pero **solo genera base y matrices del año seleccionado**.

In [3]:
manifest = build_outputs(
    PROJECT_ROOT,
    year=ANIO_ANALISIS,
    min_age=EDAD_MINIMA,
    valid_years=ANIOS_VALIDOS,
    execution_context="notebook_12_parametrizado_proceso_python_limpio_sin_nbclient",
    inspect_compatibility=True,
)

print("Estado global:", manifest["validacion"]["estado_global"])
print("Notebook ejecutado:", manifest["validacion"]["notebook_ejecutado"])
print("Compatibilidad inspeccionada:", manifest["validacion"]["compatibilidad_anios_inspeccionada"])
print("Dimensiones:", manifest["dimensiones"])

Estado global: ok
Notebook ejecutado: True
Compatibilidad inspeccionada: True
Dimensiones: {'fuente_filas': 1203231, 'universo_filas': 141579, 'hogares_unicos': 80872, 'X_columnas': 67, 'continuas': 6, 'categoricas': 11}


## Universo analítico

Indicador del universo:

$$
\mathbb{1}(U_i) =
\mathbb{1}(\text{anio}_i = a)\,
\mathbb{1}(\text{edad}_i \ge e_{\min})\,
\mathbb{1}(y_i > 0)
$$

donde \(a =\) `ANIO_ANALISIS`, \(e_{\min} =\) `EDAD_MINIMA` y \(y_i\) es `ingreso_persona_laboral_negocio_tri`.

Los resultados futuros con esta base describen asociaciones **entre adultos con ingreso laboral/de negocio positivo**. No explican quién tiene ingreso positivo ni corrigen sesgos de selección.

In [4]:
flujo = pd.read_csv(TABLE_DIR / "flujo_universo.csv")
mostrar(flujo)

              paso                criterio  filas_antes  excluidas  filas_despues  hogares_unicos_despues
0        anio_2024            anio == 2024      1203231     894633         308598                   91414
1      edad_valida    edad valida y finita       308598          0         308598                   91414
2          adultos              edad >= 18       308598      89770         218828                   91389
3    target_valido  target valido y finito       218828          0         218828                   91389
4  target_positivo              target > 0       218828      77249         141579                   80872


## Reproducción de 2024

Cuando `ANIO_ANALISIS = 2024`, se compara la ejecución actual contra los conteos y métricas principales de la etapa 12 original. Las tolerancias son explícitas: cero para conteos/dimensiones y 0.01 pesos para métricas monetarias.

In [5]:
reproduccion_rows = []
if ANIO_ANALISIS == 2024:
    target_no_pond = pd.read_csv(TABLE_DIR / "target_resumen_no_ponderado.csv").iloc[0]
    rango = pd.read_csv(TABLE_DIR / "dependencia_rango_matriz.csv").iloc[0]
    esperados = {
        "universo_filas": (141579, manifest["dimensiones"]["universo_filas"], 0),
        "hogares_unicos": (80872, manifest["dimensiones"]["hogares_unicos"], 0),
        "X_columnas": (67, manifest["dimensiones"]["X_columnas"], 0),
        "rango_con_intercepto": (68, int(rango["rango_con_intercepto"]), 0),
        "media_target": (31171.869714929475, float(target_no_pond["media"]), 0.01),
        "mediana_target": (24245.89, float(target_no_pond["mediana"]), 0.01),
        "p99_target": (154663.03, float(target_no_pond["p99"]), 0.01),
        "max_target": (17021739.12, float(target_no_pond["max"]), 0.01),
    }
    for metrica, (esperado, observado, tolerancia) in esperados.items():
        diferencia = abs(observado - esperado)
        reproduccion_rows.append(
            {
                "metrica": metrica,
                "esperado": esperado,
                "observado": observado,
                "tolerancia": tolerancia,
                "diferencia_abs": diferencia,
                "resultado": "ok" if diferencia <= tolerancia else "revisar",
            }
        )
else:
    reproduccion_rows.append(
        {
            "metrica": "reproduccion_2024",
            "esperado": "solo aplica con ANIO_ANALISIS=2024",
            "observado": ANIO_ANALISIS,
            "tolerancia": "",
            "diferencia_abs": "",
            "resultado": "no_aplica",
        }
    )

reproduccion = pd.DataFrame(reproduccion_rows)
reproduccion.to_csv(TABLE_DIR / "reproduccion_2024.csv", index=False, encoding="utf-8")
mostrar(reproduccion, n=20)

if ANIO_ANALISIS == 2024 and not reproduccion["resultado"].eq("ok").all():
    raise AssertionError("La reproducción 2024 no quedó dentro de tolerancias.")

                metrica        esperado       observado  tolerancia  diferencia_abs resultado
0        universo_filas    141,579.0000    141,579.0000      0.0000          0.0000        ok
1        hogares_unicos     80,872.0000     80,872.0000      0.0000          0.0000        ok
2            X_columnas         67.0000         67.0000      0.0000          0.0000        ok
3  rango_con_intercepto         68.0000         68.0000      0.0000          0.0000        ok
4          media_target     31,171.8697     31,171.8697      0.0100          0.0000        ok
5        mediana_target     24,245.8900     24,245.8900      0.0100          0.0000        ok
6            p99_target    154,663.0300    154,663.0300      0.0100          0.0000        ok
7            max_target 17,021,739.1200 17,021,739.1200      0.0100          0.0000        ok


## Target

El target se conserva en escala original nominal trimestral. El logaritmo se calcula solo como diagnóstico visual:

$$
\log(y_i)
$$

No reemplaza al target activo.

In [6]:
target_no_pond = pd.read_csv(TABLE_DIR / "target_resumen_no_ponderado.csv")
target_pond = pd.read_csv(TABLE_DIR / "target_resumen_ponderado.csv")
target_grupos = pd.read_csv(TABLE_DIR / "target_por_region_sexo_escolaridad.csv")

mostrar(target_no_pond)
mostrar(target_pond)
mostrar(target_grupos.sort_values(["variable", "mediana"], ascending=[True, False]), n=30)

        metrica       n       media     mediana    desv_std    min      p01        p05        p10         p25         p50         p75         p90         p95  \
0  no_ponderado  141579 31,171.8697 24,245.8900 64,171.3549 2.9300 295.0800 1,516.3000 3,815.2100 12,433.4400 24,245.8900 38,225.2700 59,016.3800 79,239.1300   

           p99             max  
0 154,663.0300 17,021,739.1200  
                        metrica  n_muestral     suma_factor  media_ponderada      p01        p05        p10         p25  mediana_ponderada         p75  \
0  ponderado_factor_descriptivo      141579 61,105,085.0000      33,133.2329 342.3900 1,770.4809 4,402.1700 13,694.1243        25,081.9500 39,836.0400   

          p90         p95          p99             max  
0 62,853.2400 87,540.9600 169,472.5488 17,021,739.1200  
           variable                                categoria      n       media     mediana     desv_std  media_ponderada_factor  mediana_ponderada_factor
6   nivelaprob_desc              

## Figuras del año seleccionado

Las figuras se leen desde `reports/figures/preparacion_determinantes/<anio>/`.

In [7]:
for nombre in [
    "target_hist_original_log.png",
    "target_log_por_region.png",
    "target_log_por_sexo.png",
    "target_log_por_escolaridad.png",
]:
    ruta = FIG_DIR / nombre
    print(ruta.relative_to(PROJECT_ROOT))
    if Image is not None and display is not None:
        display(Image(filename=str(ruta)))

reports\figures\preparacion_determinantes\2024\target_hist_original_log.png
<IPython.core.display.Image object>
reports\figures\preparacion_determinantes\2024\target_log_por_region.png
<IPython.core.display.Image object>
reports\figures\preparacion_determinantes\2024\target_log_por_sexo.png
<IPython.core.display.Image object>
reports\figures\preparacion_determinantes\2024\target_log_por_escolaridad.png
<IPython.core.display.Image object>


## Predictores iniciales y faltantes

La selección inicial permanece fija y parsimoniosa. `factor`, `factor_hogar`, `est_dis` y `upm` se conservan como metadata; no entran como predictores. `est_socio` y `tam_emp_principal_desc` siguen pendientes/diagnósticos.

In [8]:
predictores = pd.read_csv(TABLE_DIR / "predictores_iniciales.csv")
faltantes = pd.read_csv(TABLE_DIR / "faltantes_predictores.csv")
diccionario = pd.read_csv(TABLE_DIR / "diccionario_variables.csv")

mostrar(predictores, n=40)
mostrar(faltantes, n=40)

                   variable        tipo              papel                                     transformacion                  referencia_o_tratamiento
0                      edad    continua  predictor inicial  sin escalar; estandarizada solo en matriz diag...                                       NaN
1                n_trabajos    continua  predictor inicial  sin escalar; estandarizada solo en matriz diag...                                       NaN
2      horas_trabajos_total    continua  predictor inicial  sin escalar; estandarizada solo en matriz diag...                                       NaN
3                 tot_integ    continua  predictor inicial  sin escalar; estandarizada solo en matriz diag...                                       NaN
4                   menores    continua  predictor inicial  sin escalar; estandarizada solo en matriz diag...                                       NaN
5                    p65mas    continua  predictor inicial  sin escalar; estandarizada s

## Codificación one-hot

Para cada variable categórica se usa one-hot con \(k-1\) columnas y una referencia explícita:

$$
D_{ic} = \mathbb{1}(x_i = c), \quad c \ne c_{\mathrm{ref}}
$$

No se guarda intercepto. Si la referencia prevista no aparece en otro año, la ejecución debe detenerse o quedar marcada para revisión; no se escoge otra referencia silenciosamente.

In [9]:
referencias = pd.read_csv(TABLE_DIR / "referencias_ohe.csv")
mapping_ohe = pd.read_csv(TABLE_DIR / "mapping_ohe.csv")

mostrar(referencias, n=30)
print("Categorías documentadas en mapping:", len(mapping_ohe))
print("Dummies activas:", int(mapping_ohe["dummy"].fillna("").ne("").sum()))

                   variable                                referencia                                           criterio  n_referencia
0                 sexo_desc                                    Hombre  categoria interpretable definida antes de modelar         82304
1           nivelaprob_desc                                   Ninguno  categoria interpretable definida antes de modelar          4145
2            region_banxico                                    Centro  categoria interpretable definida antes de modelar         35628
3              tam_loc_desc  Localidades con 100 000 y más habitantes  categoria interpretable definida antes de modelar         54544
4           parentesco_desc                                   Jefe(a)  categoria interpretable definida antes de modelar         68035
5             hablaind_desc                                        No  categoria interpretable definida antes de modelar        130582
6               segsoc_desc                            

## Estandarización diagnóstica

La matriz sin escalar es la referencia principal. Además se genera una versión diagnóstica donde solo las continuas se transforman como:

$$
z_i = \frac{x_i - \bar{x}}{s_x}
$$

Estos parámetros se calculan sobre la muestra completa solo para diagnóstico. En una etapa futura, si se crean train/test, el escalamiento debe ajustarse únicamente con entrenamiento.

In [10]:
params = pd.read_csv(TABLE_DIR / "parametros_estandarizacion.csv")
mostrar(params)

               variable   media  sd_muestral_ddof1  constante                                      formula
0                  edad 40.9757            14.3748      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1
1            n_trabajos  1.0555             0.3549      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1
2  horas_trabajos_total 43.7648            19.8207      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1
3             tot_integ  4.0133             1.8925      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1
4               menores  0.6973             0.9592      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1
5                p65mas  0.2597             0.5613      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1


## Asociaciones exploratorias con el target

Continuas:

$$
\rho_s = \mathrm{corr}(R(x), R(y))
$$

Dummies:

$$
r_{pb} = \mathrm{corr}(D, y)
$$

El punto biserial se calcula como Pearson entre un indicador 0/1 y el ingreso. Cada dummy compara una categoría contra el resto. Estas asociaciones son marginales, no ponderadas, sin p-values y no implican causalidad.

In [11]:
cont_assoc = pd.read_csv(TABLE_DIR / "correlaciones_continuas_spearman.csv")
bin_assoc = pd.read_csv(TABLE_DIR / "correlaciones_dummies_punto_biserial.csv")

mostrar(
    cont_assoc.assign(abs_rho=cont_assoc["spearman_ingreso_original"].abs())
    .sort_values("abs_rho", ascending=False)
    .drop(columns="abs_rho"),
    n=20,
)
mostrar(
    bin_assoc.assign(abs_r=bin_assoc["punto_biserial_ingreso_original"].abs())
    .sort_values("abs_r", ascending=False)
    .drop(columns="abs_r"),
    n=20,
)

               variable      tipo  n_valido  spearman_ingreso_original                                               nota
2  horas_trabajos_total  continua    141579                     0.3434  Correlacion de rangos exploratoria no ponderad...
5                p65mas  continua    141579                    -0.1394  Correlacion de rangos exploratoria no ponderad...
1            n_trabajos  continua    141579                     0.0853  Correlacion de rangos exploratoria no ponderad...
3             tot_integ  continua    141579                    -0.0534  Correlacion de rangos exploratoria no ponderad...
0                  edad  continua    141579                    -0.0446  Correlacion de rangos exploratoria no ponderad...
4               menores  continua    141579                    -0.0076  Correlacion de rangos exploratoria no ponderad...
                                                dummy        variable_original                                 categoria_vs_resto  n_valido  frecu

## Dependencia entre predictores

Se revisan constantes, duplicados, correlaciones entre continuas, asociación entre categóricas, rango de matriz y VIF:

$$
VIF_j = \frac{1}{1 - R_j^2}
$$

El VIF depende de la codificación y es un diagnóstico de dependencia, no una regla automática de eliminación.

In [12]:
rango = pd.read_csv(TABLE_DIR / "dependencia_rango_matriz.csv")
constantes = pd.read_csv(TABLE_DIR / "dependencia_constantes.csv")
duplicadas = pd.read_csv(TABLE_DIR / "dependencia_duplicadas.csv")
vif = pd.read_csv(TABLE_DIR / "dependencia_vif.csv")
cramers = pd.read_csv(TABLE_DIR / "dependencia_categoricas_cramers_v.csv")

mostrar(rango)
print("Constantes:", len(constantes))
print("Duplicadas:", len(duplicadas))
mostrar(vif.sort_values("vif", ascending=False), n=20)
mostrar(cramers.sort_values("cramers_v", ascending=False), n=15)

    filas  columnas_X  columnas_con_intercepto  rango_con_intercepto  deficiencia_rango          estado
0  141579          67                       68                    68                  0  rango_completo
Constantes: 0
Duplicadas: 0
                                              columna     vif estado                                               nota
0   contrato_principal_desc__no_aplica_sin_contrat... 36.4987     ok  VIF_j = 1/(1-R_j^2). En dummies depende de la ...
1                            subor_principal_desc__si 36.3891     ok  VIF_j = 1/(1-R_j^2). En dummies depende de la ...
2                         nivelaprob_desc__secundaria 12.5976     ok  VIF_j = 1/(1-R_j^2). En dummies depende de la ...
3        nivelaprob_desc__preparatoria_o_bachillerato 11.8357     ok  VIF_j = 1/(1-R_j^2). En dummies depende de la ...
4   nivelaprob_desc__licenciatura_o_ingenieria_pro... 11.7787     ok  VIF_j = 1/(1-R_j^2). En dummies depende de la ...
5                           nivelaprob_desc_

## Compatibilidad inspeccionada para otros años

Esta inspección revisa disponibilidad de columnas, universo elegible, categorías observadas y referencias OHE esperadas para todos los años válidos. No genera bases ni matrices para los años distintos a `ANIO_ANALISIS`.

La comparabilidad de coeficientes entre años requerirá una especificación común posterior; esta etapa no la da por resuelta.

In [13]:
compat_anios = pd.read_csv(TABLE_ROOT / "compatibilidad_anios.csv")
compat_cats = pd.read_csv(TABLE_ROOT / "compatibilidad_categorias_ohe.csv")

mostrar(compat_anios, n=20)
mostrar(
    compat_cats[
        (compat_cats["referencia_presente"].astype(str).str.lower() != "true")
        | compat_cats["categorias_nuevas_vs_anio_base"].notna()
        | compat_cats["categorias_ausentes_vs_anio_base"].notna()
    ],
    n=40,
)

   anio                         estado  filas_anio  filas_universo  hogares_unicos  columnas_requeridas_faltantes referencias_ohe_faltantes  \
0  2018   compatible_para_generar_base      269206          120054           67807                            NaN                       NaN   
1  2020  revisar_antes_de_generar_base      315743          139394           79365                            NaN               segsoc_desc   
2  2022  revisar_antes_de_generar_base      309684          141514           80217                            NaN               segsoc_desc   
3  2024   compatible_para_generar_base      308598          141579           80872                            NaN                       NaN   

   anio_base_comparacion  categorias_nuevas_vs_anio_base  categorias_ausentes_vs_anio_base                                               nota  
0                   2024                               5                                 9  Inspeccion de esquema; no genera matrices ni b..

## Validaciones y salidas

Las validaciones comprueban universo, llave única, dimensiones consistentes entre `X`, `y` y metadata, ausencia de target/metadata en `X`, OHE con referencias, valores finitos y equivalencia del punto biserial con Pearson en un ejemplo pequeño.

In [14]:
validaciones = pd.read_csv(TABLE_DIR / "validaciones_preparacion.csv")
with open(TABLE_DIR / "manifest_preparacion_determinantes.json", encoding="utf-8") as f:
    manifest_archivo = json.load(f)

base_anios = pd.read_csv(PROCESSED_DIR / f"base_interpretable_personas_{ANIO_ANALISIS}.csv.gz", usecols=["anio"])
metadata_anios = pd.read_csv(PROCESSED_DIR / f"metadata_personas_{ANIO_ANALISIS}.csv.gz", usecols=["anio"])
salidas_anio = pd.DataFrame(
    [
        {
            "salida": "base_interpretable",
            "anios_presentes": ",".join(map(str, sorted(base_anios["anio"].unique()))),
            "resultado": "ok" if set(base_anios["anio"].unique()) == {ANIO_ANALISIS} else "revisar",
        },
        {
            "salida": "metadata",
            "anios_presentes": ",".join(map(str, sorted(metadata_anios["anio"].unique()))),
            "resultado": "ok" if set(metadata_anios["anio"].unique()) == {ANIO_ANALISIS} else "revisar",
        },
    ]
)
salidas_anio.to_csv(TABLE_DIR / "salidas_anio_unico.csv", index=False, encoding="utf-8")

mostrar(validaciones, n=40)
mostrar(salidas_anio)
print("Estado manifest:", manifest_archivo["validacion"])
print("Directorio de bases fuera de Git:", PROCESSED_DIR)
print("Directorio de tablas versionadas:", TABLE_DIR)
print("Directorio de figuras versionadas:", FIG_DIR)

if not validaciones["resultado"].fillna("ok").eq("ok").all():
    raise AssertionError("Hay validaciones de preparación en estado revisar.")
if not salidas_anio["resultado"].eq("ok").all():
    raise AssertionError("Alguna salida contiene años distintos al configurado.")

                                   validacion resultado                                            detalle  valor_scipy  valor_pearson  diferencia_abs  \
0                   universo_anio_edad_target        ok  Todas las filas finales cumplen anio 2024, eda...          NaN            NaN             NaN   
1                 salidas_universo_anio_unico        ok  Anios presentes en universo/base interpretable...          NaN            NaN             NaN   
2                         llave_persona_unica        ok  Duplicados ['anio', 'folioviv', 'foliohog', 'n...          NaN            NaN             NaN   
3                          filas_X_y_metadata        ok  Dimensiones: X=(141579, 67), X_scaled=(141579,...          NaN            NaN             NaN   
4          sin_target_derivados_metadata_en_X        ok                  Columnas prohibidas en X: ninguna          NaN            NaN             NaN   
5                ohe_referencias_consistentes        ok  Cada variable categ

## Requisitos para modelos futuros

- Los notebooks de modelado deberán usar `ANIO_ANALISIS` explícito.
- Leerán exclusivamente la base del año seleccionado.
- Guardarán modelos, métricas y figuras separados por año y especificación.
- No mezclarán años automáticamente.
- Los preprocesadores aprendidos se ajustarán con entrenamiento cuando se defina una evaluación predictiva.

Pendientes metodológicos que siguen abiertos:

- Aprobar estrategia inferencial.
- Definir partición futura considerando hogares.
- Ajustar codificador y escalador solo con entrenamiento si se crean particiones.
- Decidir si `est_socio` entra como predictor o permanece como diagnóstico.
- Decidir si `tam_emp_principal_desc` se recodifica o permanece fuera por redundancia estructural.
- Revisar variables pendientes con etiquetas ambiguas antes de ampliar `X`.